In [1]:
import pandas as pd

app = pd.read_csv("data/application_record.csv")
credit = pd.read_csv("data/credit_record.csv")

print("application_record:", app.shape)
print("credit_record:", credit.shape)
app.info()

application_record: (438557, 18)
credit_record: (1048575, 3)
<class 'pandas.DataFrame'>
RangeIndex: 438557 entries, 0 to 438556
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   438557 non-null  int64  
 1   CODE_GENDER          438557 non-null  str    
 2   FLAG_OWN_CAR         438557 non-null  str    
 3   FLAG_OWN_REALTY      438557 non-null  str    
 4   CNT_CHILDREN         438557 non-null  int64  
 5   AMT_INCOME_TOTAL     438557 non-null  float64
 6   NAME_INCOME_TYPE     438557 non-null  str    
 7   NAME_EDUCATION_TYPE  438557 non-null  str    
 8   NAME_FAMILY_STATUS   438557 non-null  str    
 9   NAME_HOUSING_TYPE    438557 non-null  str    
 10  DAYS_BIRTH           438557 non-null  int64  
 11  DAYS_EMPLOYED        438557 non-null  int64  
 12  FLAG_MOBIL           438557 non-null  int64  
 13  FLAG_WORK_PHONE      438557 non-null  int64  
 14  FLAG_PHONE        

In [2]:
print(app.isnull().sum())
print(app['ID'].duplicated().sum())
print(credit['STATUS'].value_counts())

ID                          0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_MOBIL                  0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        134203
CNT_FAM_MEMBERS             0
dtype: int64
47
STATUS
C    442031
0    383120
X    209230
1     11090
5      1693
2       868
3       320
4       223
Name: count, dtype: int64


In [3]:
# Define which statuses count as "risky" (30+ days overdue)
risky_statuses = ['1', '2', '3', '4', '5']

# For each applicant, check if they were EVER risky across all months
credit['IS_RISKY'] = credit['STATUS'].isin(risky_statuses).astype(int)

# Collapse to one row per applicant: if risky in ANY month, mark as risky overall
target = credit.groupby('ID')['IS_RISKY'].max().reset_index()
target.rename(columns={'IS_RISKY': 'TARGET'}, inplace=True)

print(target.shape)
print(target['TARGET'].value_counts())

(45985, 2)
TARGET
0    40635
1     5350
Name: count, dtype: int64


In [4]:
# Merge application data with target label (inner join keeps only applicants with known outcome)
df = pd.merge(app, target, on='ID', how='inner')

print(df.shape)
df.head()
print(df['TARGET'].value_counts())

(36457, 19)
TARGET
0    32166
1     4291
Name: count, dtype: int64


In [5]:
df.to_csv("data/merged_data.csv", index=False)
print("Saved successfully")

Saved successfully
